### The DeepRitz-Method

Consider the following PDE:
\begin{align*}
    -\Delta u + 10 u &= -1 \text{ in } \Omega, \\
    u &= 0 \text{ on } \partial \Omega,
\end{align*} 

with $\Omega=[0, 1] \times [0, 1]$.

#### Task 1: Derive an energy functional for this PDE.

Hint: Consider the functional 
\begin{equation}
    F(v):= \int_\Omega \vert v(x) \vert^2 dx
\end{equation}
and compute its derivative at $u$ into direction $v$, that is, 
\begin{equation}
\lim_{\varepsilon\to 0} \varepsilon^{-1}(F(u+\varepsilon v) - F(u)) = ???.
\end{equation}

#### Task 2: Similar to ex1_ritz.ipynb, solve the PDE using the DeepRitz method. Below you find the code for solving ex1_ritz. Adapt it to the modified PDE above.
Compare your solution with the solution of the previous exercise ex1_ritz. What do you observe?

In [ ]:
import torchphysics as tp 
import torch
import pytorch_lightning as pl

X = tp.spaces.R2('x') 
U = tp.spaces.R1('u')

In [ ]:
# Implement the square [0, 1] x [0, 1] 
square = tp.domains.Parallelogram(X, [0, 0], [1, 0], [0, 1])

In [ ]:
# Here we create the sampler. For DeepRitz we need to approximate the integral, therefore one usually needs more points
# then in the PINN approach to get a good estimate.
# But since in the energy functional the derivatives are of lower order, this generally does not lead to memory problems.
inner_sampler = tp.samplers.RandomUniformSampler(square, n_points=100000) 

bound_sampler = tp.samplers.RandomUniformSampler(square.boundary, n_points=40000)

The classic DeepRitz method uses a ResNet-architecture instead of a FCN. This is also implemented and used here: 

In [ ]:
# Add the input and output space
model = tp.models.DeepRitzNet(input_space=X, output_space=U, width=20, depth=4)

The transformation of the mathematical equations is handled similar to the PINN approach. For each integral in the energy functional, we define a condition.

Instead of a *PINNCondition*, we now use a *DeepRitzCondition*. While the *PINNCondition* computes the mean of the squared outputs of the residual function, the *DeepRitzCondition* only averages the output of the integrand function (without computing squares, which allows negative losses).

In [ ]:
# The integral for the boundary
def bound_residual(u):
    return u**2

bound_cond = tp.conditions.DeepRitzCondition(module=model, sampler=bound_sampler, 
                                             integrand_fn=bound_residual, weight=100)

In [ ]:
# The integrand function for the integral over Omega.
### TODO: Implement the integrand of the integral over Omega of your energy functional. 
###       Below you can see this for the normal Poisson equation from the previous excercise.
def energy_residual(u, x):
    grad_term = torch.sum(tp.utils.grad(u, x)**2, dim=1, keepdim=True)
    return 0.5*grad_term + u

pde_cond = tp.conditions.DeepRitzCondition(module=model, sampler=inner_sampler, 
                                           integrand_fn=energy_residual, weight=1.0)

In [ ]:
learning_rate = 1e-3
training_iterations = 2000

optim = tp.OptimizerSetting(optimizer_class=torch.optim.Adam, lr=learning_rate)
solver = tp.solver.Solver(train_conditions=[bound_cond, pde_cond], optimizer_setting=optim)

trainer = pl.Trainer(devices=1, accelerator="gpu",
                     max_steps=training_iterations,
                     logger=False,
                     benchmark=True,
                     enable_checkpointing=False)

trainer.fit(solver)

In [ ]:
plot_sampler = tp.samplers.PlotSampler(plot_domain=square, n_points=640, device='cuda')
fig = tp.utils.plot(model, lambda u : u, plot_sampler, plot_type='contour_surface')